# Predicting Student Health Risk - Baseline Modeling

This notebook builds the first public, fold-safe baseline for S6E7. The emphasis is reliability: robust input discovery, explicit preprocessing, stratified validation, class-balanced diagnostics, and a valid `submission.csv`.

This is deliberately not the final model. It is the reference point that later LightGBM, CatBoost, XGBoost, feature engineering, and ensembling work must beat.

## 1. Setup

The baseline uses sklearn only so it is portable and easy to debug. Missing values are imputed and missingness indicators are preserved because missingness is substantial in both train and test.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

SEED = 42
N_SPLITS = 5

candidate_dirs = [
    Path('/kaggle/input/playground-series-s6e7'),
    Path('/kaggle/input/predicting-student-health-risk'),
    Path('../data'),
    Path('data'),
]
DATA_DIR = next(
    (path for path in candidate_dirs if (path / 'train.csv').exists()),
    None,
)
if DATA_DIR is None and Path('/kaggle/input').exists():
    for train_path in Path('/kaggle/input').glob('**/train.csv'):
        parent = train_path.parent
        if (parent / 'test.csv').exists() and (parent / 'sample_submission.csv').exists():
            DATA_DIR = parent
            break
if DATA_DIR is None:
    raise FileNotFoundError('Could not locate train/test/sample_submission CSV files.')

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('..')
PRED_DIR = OUTPUT_DIR / 'predictions'
PRED_DIR.mkdir(exist_ok=True)
print(f'Using data directory: {DATA_DIR}')

## 2. Load Data And Define The Target

The target is the only train column that does not appear in test: `health_condition`. The sample submission defines the ID column and output column order.

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

id_col = sample_submission.columns[0]
target_candidates = [col for col in train.columns if col not in test.columns]
target = target_candidates[0]

feature_cols = [col for col in test.columns if col in train.columns and col != id_col]
X = train[feature_cols]
y = train[target]
X_test = test[feature_cols]

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [col for col in feature_cols if col not in num_cols]
classes = np.array(sorted(y.unique()))

print(f'train shape: {train.shape}')
print(f'test shape: {test.shape}')
print(f'target: {target}')
print(f'numeric features ({len(num_cols)}): {num_cols}')
print(f'categorical features ({len(cat_cols)}): {cat_cols}')
display(y.value_counts(normalize=True).to_frame('target_share'))

## 3. Validation Design

The target is imbalanced, so we use stratified folds and track class-balanced metrics. Accuracy is shown for orientation, but macro F1 and balanced accuracy are more informative for minority-class behavior.

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
metric_rows = []
oof_pred = pd.Series(index=train.index, dtype=object)
test_votes = []

print(f'Using {N_SPLITS}-fold StratifiedKFold with seed={SEED}.')

## 4. Preprocessing And Baseline Model

Numeric and categorical features are handled separately:

- numeric fields use median imputation plus missing-value indicators;
- categorical fields use most-frequent imputation and ordinal encoding;
- the model uses class-balanced weighting to avoid simply learning the majority `at-risk` class.

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median', add_indicator=True), num_cols),
        (
            'cat',
            Pipeline(
                [
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    (
                        'encoder',
                        OrdinalEncoder(
                            handle_unknown='use_encoded_value',
                            unknown_value=-1,
                        ),
                    ),
                ]
            ),
            cat_cols,
        ),
    ],
    remainder='drop',
)

model = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=160,
    l2_regularization=0.05,
    early_stopping=True,
    class_weight='balanced',
    random_state=SEED,
)

pipeline = Pipeline(
    [
        ('preprocess', preprocess),
        ('model', model),
    ]
)

## 5. Cross-Validated Training

Each fold fits the full preprocessing pipeline only on the training fold. This keeps imputation and category handling fold-safe. Test predictions are collected from every fold and combined by majority vote for the first submission.

In [ ]:
for fold, (trn_idx, val_idx) in enumerate(cv.split(X, y), start=1):
    X_trn, X_val = X.iloc[trn_idx], X.iloc[val_idx]
    y_trn, y_val = y.iloc[trn_idx], y.iloc[val_idx]

    pipeline.fit(X_trn, y_trn)
    val_pred = pipeline.predict(X_val)
    test_votes.append(pipeline.predict(X_test))
    oof_pred.iloc[val_idx] = val_pred

    row = {
        'fold': fold,
        'accuracy': accuracy_score(y_val, val_pred),
        'balanced_accuracy': balanced_accuracy_score(y_val, val_pred),
        'macro_f1': f1_score(y_val, val_pred, average='macro'),
        'weighted_f1': f1_score(y_val, val_pred, average='weighted'),
    }
    metric_rows.append(row)
    print(
        f"Fold {fold}: "
        f"accuracy={row['accuracy']:.5f}, "
        f"balanced_accuracy={row['balanced_accuracy']:.5f}, "
        f"macro_f1={row['macro_f1']:.5f}"
    )

results = pd.DataFrame(metric_rows)
display(results)
display(results.mean(numeric_only=True).to_frame('mean'))

## 6. Error Diagnostics

The confusion matrix and per-class report tell us whether the model is buying minority-class recall at an unacceptable precision cost. This is the main diagnostic before trying stronger GBDT models.

In [ ]:
print(classification_report(y, oof_pred, digits=4))

cm = pd.DataFrame(
    confusion_matrix(y, oof_pred, labels=classes),
    index=[f'true_{cls}' for cls in classes],
    columns=[f'pred_{cls}' for cls in classes],
)
display(cm)

pd.DataFrame(
    {
        id_col: train[id_col] if id_col in train else train.index,
        'oof_pred': oof_pred,
        'target': y,
    }
).to_csv(PRED_DIR / 'hgb_oof_predictions.csv', index=False)

## 7. Create The First Submission

For the baseline submission, each fold votes for a class on every test row. Later notebooks should replace this with averaged probabilities from calibrated LightGBM/CatBoost/XGBoost models.

In [ ]:
vote_frame = pd.DataFrame(np.column_stack(test_votes))
test_pred = vote_frame.mode(axis=1)[0]

submission = sample_submission.copy()
submission.iloc[:, 1] = test_pred.values
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)

display(submission.head())
display(submission.iloc[:, 1].value_counts(normalize=True).to_frame('prediction_share'))
print(f'Wrote submission to {OUTPUT_DIR / "submission.csv"}')

## 8. Next Modeling Moves

- Compare this class-balanced HGB baseline against CatBoost with native categorical handling.
- Add missingness and lifestyle interaction features in a controlled feature-family test.
- Save OOF probability matrices for blend and calibration work.
- Use leaderboard submissions only after validation and prediction-share checks look plausible.


## 9. Public Version 3 Output Review

The public v3 notebook completed successfully and produced the first leaderboard calibration submission.

**Validation and leaderboard summary**

| Signal | Value |
| --- | ---: |
| OOF accuracy | `0.8785` |
| OOF balanced accuracy | `0.9091` |
| OOF macro F1 | `0.7610` |
| Public leaderboard score | `0.90603` |

**What the model learned**

The class-balanced HGB model is deliberately aggressive about recovering minority classes. It reaches high recall for `fit` and `unhealthy`, but precision is much weaker for those same classes. That means the model is useful as a class-balance probe, not yet as the champion.

| Class | OOF recall | OOF precision |
| --- | ---: | ---: |
| `at-risk` | `0.8701` | `0.9875` |
| `fit` | `0.9226` | `0.5055` |
| `unhealthy` | `0.9345` | `0.5657` |

**Prediction mix**

| Split | `at-risk` | `fit` | `unhealthy` |
| --- | ---: | ---: | ---: |
| Train target | `85.87%` | `5.77%` | `8.36%` |
| OOF predictions | `75.65%` | `10.53%` | `13.82%` |
| Test submission | `75.71%` | `10.39%` | `13.90%` |

**Decision**

This submission is worth keeping as the first public calibration point. The next notebook should test a less aggressive prior strategy: unweighted or lightly weighted CatBoost/LightGBM/HGB, then compare by public score, OOF accuracy, macro F1, and prediction mix.
